In [2]:
# Install packages (run once)
!pip install sage-data-client pandas matplotlib

                            timestamp             name  value  \
0 2026-05-22 23:31:00.536618750+00:00  env.temperature  40.69   
1 2026-05-22 23:31:29.360984631+00:00  env.temperature  40.70   
2 2026-05-22 23:31:58.943212556+00:00  env.temperature  40.72   
3 2026-05-22 23:32:28.873584032+00:00  env.temperature  40.73   
4 2026-05-22 23:33:00.507131072+00:00  env.temperature  40.74   

                    meta.host   meta.job         meta.node  \
0  000048b02d05a1c2.ws-nxcore  Pluginctl  000048b02d05a1c2   
1  000048b02d05a1c2.ws-nxcore  Pluginctl  000048b02d05a1c2   
2  000048b02d05a1c2.ws-nxcore  Pluginctl  000048b02d05a1c2   
3  000048b02d05a1c2.ws-nxcore  Pluginctl  000048b02d05a1c2   
4  000048b02d05a1c2.ws-nxcore  Pluginctl  000048b02d05a1c2   

               meta.plugin meta.sensor       meta.task meta.vsn meta.zone  
0  waggle/plugin-iio:0.6.0      bme280  wes-iio-bme280     W02C      core  
1  waggle/plugin-iio:0.6.0      bme280  wes-iio-bme280     W02C      core  
2  waggl

In [ ]:
# Example: Access SAGE Continuum environmental data

from sage_data_client import query
import pandas as pd

# Example query:
# Fetch temperature sensor data from the last 24 hours

df = query(
    start="-24h",
    filter={
        "name": "env.temperature"
    }
)

print(df.head())
print(df.columns)

In [ ]:
# ============================================================
# SAGE CONTINUUM DATA COLLECTION SCRIPT
# ------------------------------------------------------------
# Collects data every hour for 7 days
# Saves separate numbered CSV files into:
# E:\Sage_data
#
# Each file contains:
# - timestamp
# - node id (vsn)
# - zone
# - person count
# - car count
# - temperature
# - humidity
# - wind
# - rain
#
# Strategy:
# - Query last 1 hour
# - Split into 10 bins (~6 min each)
# - Average values inside each bin
# - Save one CSV per hour
# ============================================================

# Install once
# !pip install sage-data-client pandas numpy

import os
import time
import pandas as pd
import numpy as np

from sage_data_client import query

# ============================================================
# CONFIG
# ============================================================

SAVE_DIR = r"E:\Sage_data"

# create folder if missing
os.makedirs(SAVE_DIR, exist_ok=True)

# total runtime
DAYS = 7

# collect every hour
SLEEP_SECONDS = 60 * 60

# total iterations
TOTAL_HOURS = DAYS * 24

# 10 bins per hour
NUM_BINS = 10

# ============================================================
# SIGNALS TO COLLECT
# ============================================================

SIGNALS = {
    "person": "person",
    "car": "car",
    "temperature": "env.temperature",
    "humidity": "env.humidity",
    "wind": "env.wind.speed",
    "rain": "env.rain"
}

# ============================================================
# HELPER FUNCTION
# ============================================================

def get_signal_data(signal_name):

    try:

        df = query(
            start="-1h",
            filter={
                "name": signal_name
            }
        )

        return df

    except Exception as e:

        print("ERROR:", signal_name, e)

        return pd.DataFrame()

# ============================================================
# MAIN COLLECTION LOOP
# ============================================================

file_num = 0

for hour_idx in range(TOTAL_HOURS):

    print("=" * 60)
    print("COLLECTION ROUND:", hour_idx + 1)
    print("=" * 60)

    merged_rows = []

    # --------------------------------------------------------
    # GET DATA FOR EACH SIGNAL
    # --------------------------------------------------------

    signal_dfs = {}

    for key in SIGNALS:

        sig = SIGNALS[key]

        print("Downloading:", sig)

        df = get_signal_data(sig)

        if len(df) == 0:
            continue

        # convert timestamp
        df["timestamp"] = pd.to_datetime(df["timestamp"])

        signal_dfs[key] = df

    # --------------------------------------------------------
    # PROCESS EACH SIGNAL
    # --------------------------------------------------------

    for key in signal_dfs:

        df = signal_dfs[key]

        # keep needed cols only
        cols = ["timestamp", "value"]

        if "meta.vsn" in df.columns:
            cols.append("meta.vsn")

        if "meta.zone" in df.columns:
            cols.append("meta.zone")

        df = df[cols]

        # rename
        df = df.rename(columns={"value": key})

        # ----------------------------------------------------
        # GROUP BY NODE
        # ----------------------------------------------------

        node_groups = df.groupby("meta.vsn")

        for node_id, node_df in node_groups:

            node_df = node_df.sort_values("timestamp")

            # create bins
            node_df["bin"] = pd.cut(
                node_df["timestamp"].astype(np.int64),
                bins=NUM_BINS,
                labels=False
            )

            # aggregate
            agg_df = node_df.groupby("bin").agg({
                "timestamp": "first",
                key: "mean",
                "meta.zone": "first"
            }).reset_index(drop=True)

            agg_df["vsn"] = node_id

            merged_rows.append(
                agg_df.rename(columns={"meta.zone": "zone"})
            )

    # --------------------------------------------------------
    # MERGE ALL SIGNALS
    # --------------------------------------------------------

    if len(merged_rows) > 0:

        final_df = pd.concat(merged_rows, ignore_index=True)

        # merge same timestamps/nodes
        final_df = final_df.groupby(
            ["timestamp", "vsn", "zone"],
            as_index=False
        ).mean(numeric_only=True)

        # sort
        final_df = final_df.sort_values(
            ["vsn", "timestamp"]
        )

        # ----------------------------------------------------
        # SAVE CSV
        # ----------------------------------------------------

        save_path = os.path.join(
            SAVE_DIR,
            f"data_{file_num:04d}.csv"
        )

        final_df.to_csv(save_path, index=False)

        print("Saved:", save_path)
        print("Rows:", len(final_df))

        file_num += 1

    else:

        print("No data collected.")

    # --------------------------------------------------------
    # WAIT 1 HOUR
    # --------------------------------------------------------

    if hour_idx < TOTAL_HOURS - 1:

        print("Sleeping for 1 hour...")
        time.sleep(SLEEP_SECONDS)

print("DONE!")

In [ ]:
import pandas as pd
import glob

files = glob.glob(r"E:\Sage_data\*.csv")

dfs = []

for f in files:
    dfs.append(pd.read_csv(f))

final_df = pd.concat(dfs, ignore_index=True)

print(final_df.shape)

final_df.to_csv(r"E:\Sage_data\all_data.csv", index=False)

In [ ]:
# Example rule-based labels

df["traffic_label"] = (df["car"] > 10).astype(int)

df["human_activity"] = (df["person"] > 5).astype(int)

df["weather_event"] = (
    (df["wind"] > 15) |
    (df["rain"] > 0.2)
).astype(int)